# Assigning parcel labels to brain vertices using a volumetric atlas in MNI space

Atlases are often provided as a NIfTI file, which assigns each voxel a numeric label, and an accompanying text file (json,csv) which maps numeric to string labels.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as p
import numpy as np
import pandas as pd
import pyvista as pv

import cedalion
import cedalion.dot
import cedalion.vis.blocks as vbx
import cedalion.vis.colors
from cedalion.vis.anatomy import plot_brain_views_grid, get_vertex_colors_from_coord
import cedalion.xrutils as xrutils
from scipy.spatial import KDTree

pv.set_jupyter_backend("static")

## Loading Colin27 and ICBM-152 headmodels

In [ ]:
colin_ijk = cedalion.dot.get_standard_headmodel("colin27")
colin_inflated = cedalion.dot.get_inflated_cortex_surface("colin27")

icbm_ijk = cedalion.dot.get_standard_headmodel("icbm152")
icbm_inflated = cedalion.dot.get_inflated_cortex_surface("icbm152")

## AAL3 labels

In [ ]:
aal3_voxel_label_niftii, aal3_labels_json = cedalion.data.get_atlas_files("aal3")

# dictionary to map numeric voxel labels in nifti to string labels
with aal3_labels_json.open("r") as fin:
    aal3_num2label = json.load(fin)
    aal3_num2label = {i["index"] : i["name"] for i in aal3_num2label["labels"]}

aal3_num2label

## Brodmann labels

In [ ]:
brodmann_voxel_label_niftii, brodmann_labels_json = cedalion.data.get_atlas_files("brodmann")

# dictionary to map numeric voxel labels in nifti to string labels
with brodmann_labels_json.open("r") as fin:
    brodmann_num2label = json.load(fin)
    brodmann_num2label = {i["index"] : i["name"] for i in brodmann_num2label["labels"]}

brodmann_num2label

Brain vertex coordinates before assigning a new parcellation scheme

In [ ]:
colin_ijk.brain.vertices

The function `TwoSurfaceHeadModel.assign_parcels_via_mni_coords` applies the parcellation scheme and adds a new vertex coordinate to the brain surface.

In [ ]:
# Colin27

colin_ijk_labeled = colin_ijk.assign_parcels_via_mni_coords(
    coordinate_label="parcel_aal3",
    label_mapping=aal3_num2label,
    voxel_label_niftii=aal3_voxel_label_niftii,
    voxel_label_crs="mni152",
    mni_eps=5
)

colin_ijk_labeled = colin_ijk_labeled.assign_parcels_via_mni_coords(
    coordinate_label="parcel_brodmann",
    label_mapping=brodmann_num2label,
    voxel_label_niftii=brodmann_voxel_label_niftii,
    voxel_label_crs="mni152",
    mni_eps=5
)


colin_ijk_labeled.brain.vertices

In [ ]:
# ICBM-152

icbm_ijk_labeled = icbm_ijk.assign_parcels_via_mni_coords(
    coordinate_label="parcel_aal3",
    label_mapping=aal3_num2label,
    voxel_label_niftii=aal3_voxel_label_niftii,
    voxel_label_crs="mni152",
    mni_eps=5
)

icbm_ijk_labeled = icbm_ijk_labeled.assign_parcels_via_mni_coords(
    coordinate_label="parcel_brodmann",
    label_mapping=brodmann_num2label,
    voxel_label_niftii=brodmann_voxel_label_niftii,
    voxel_label_crs="mni152",
    mni_eps=5
)


icbm_ijk_labeled.brain.vertices

## Plotting parcellation schemes

- need to translate vertex labels to vertex colors
- different color mappings possible: mainly dicts that map string labels to a matplotlib color spec.
- helper function `get_vertex_colors_from_coord`

In [ ]:
# example: Schaefer parcel colors
schaefer_color_dict = cedalion.data.get_colin27_headmodel_files().load_parcel_colors()
display(schaefer_color_dict)

In [ ]:
vertex_colors = get_vertex_colors_from_coord(colin_ijk_labeled.brain, "parcel", color_mapping=schaefer_color_dict)
plot_brain_views_grid(colin_inflated, vertex_colors, reset_camera=True)

### Plot AAL3 Labels on Colin27

- if no color map is available, pass `None`. This generates a random color mapping.

In [ ]:
vertex_colors = get_vertex_colors_from_coord(colin_ijk_labeled.brain, "parcel_aal3", color_mapping=None)

print(len(vertex_colors))
vertex_colors[:4]

In [ ]:
plot_brain_views_grid(colin_ijk_labeled.brain, vertex_colors)
plot_brain_views_grid(colin_inflated, vertex_colors, reset_camera=True)

### Plot Brodmann Labels on Colin27

In [ ]:
vertex_colors = get_vertex_colors_from_coord(colin_ijk_labeled.brain, "parcel_brodmann", color_mapping=None)

plot_brain_views_grid(colin_ijk_labeled.brain, vertex_colors)
plot_brain_views_grid(colin_inflated, vertex_colors, reset_camera=True)

### Plot AAL3 labels on ICBM-152

In [ ]:
vertex_colors = get_vertex_colors_from_coord(icbm_ijk_labeled.brain, "parcel_aal3", color_mapping=None)

plot_brain_views_grid(icbm_ijk_labeled.brain, vertex_colors)
plot_brain_views_grid(icbm_inflated, vertex_colors, reset_camera=True)

### Plot Brodmann labels on ICBM-152

In [ ]:
vertex_colors = get_vertex_colors_from_coord(icbm_ijk_labeled.brain, "parcel_brodmann", color_mapping=None)

plot_brain_views_grid(icbm_ijk_labeled.brain, vertex_colors)
plot_brain_views_grid(icbm_inflated, vertex_colors, reset_camera=True)

## Customizing colors

Thera are multiple ways to customize the color mapping. This allows for example for selecting only specific parcels.

In [ ]:
# select only two parcels and specify colors (any matplotlib color spec works)
# parcels not contained in the mapping get the default color
aal3_color_mapping = {"Frontal_Inf_Oper_R" : "r", "Frontal_Inf_Tri_R" : "g"}

vertex_colors = get_vertex_colors_from_coord(colin_ijk_labeled.brain, "parcel_aal3", aal3_color_mapping)
plot_brain_views_grid(colin_ijk_labeled.brain, vertex_colors)


In [ ]:
# specify only a single color, that will be used for all parcels. Then select only a subset of parcels to color.

vertex_colors = get_vertex_colors_from_coord(
    colin_ijk_labeled.brain,
    "parcel_brodmann",
    color_mapping="r",
    labels=["right_BA44", "right_BA45"],
)
plot_brain_views_grid(colin_ijk_labeled.brain, vertex_colors)

## MNI Coordinate Label Check

In [ ]:
tests = np.asarray([
    [-10, -90, 0],
    [-40, -20, 50],
    [40, 20, 30],
], dtype=float)

vertex_coords = icbm_ijk_labeled.get_brain_mni152_coords().pint.dequantify().values
vertex_labels = icbm_ijk_labeled.brain.vertices.coords["parcel_brodmann"].values
_, nearest = KDTree(vertex_coords).query(tests)

for mni, vertex_idx in zip(tests, nearest):
    print(f"MNI {mni.tolist()} -> nearest surface label {vertex_labels[vertex_idx]}")

## Parcel-Level Summary Table TSVs

These compact tables have one row per existing head-model parcel, not one row per surface vertex. Medial-wall/background parcels are excluded by default because they are not real cortical parcels. The atlas label is summarized from the xarray vertex coordinate by taking the dominant label inside each parcel and keeping the vertex count/fraction.

The tables are returned as `pandas.DataFrame` objects. Use their `.to_csv` or similar methods to save them.

In [ ]:
colin_ijk_labeled.parcel_summary_from_vertex_coordinate("parcel_aal3", "colin27")

In [ ]:
colin_ijk_labeled.parcel_summary_from_vertex_coordinate("parcel_brodmann", "colin27")

In [ ]:
icbm_ijk_labeled.parcel_summary_from_vertex_coordinate("parcel_aal3", "icbm152")

In [ ]:
icbm_ijk_labeled.parcel_summary_from_vertex_coordinate("parcel_aal3", "icbm152")

## References

In [ ]:
cedalion.bib.dump_to_notebook()